In [ ]:
"""
Compile Google Street View metadata from Azure Blob Storage.

Overview
--------
This script extracts Google Street View metadata stored as individual JSON
files in Azure Blob Storage and compiles them into a single CSV dataset.

Each metadata file corresponds to one requested Street View location and
contains both the originally requested coordinates and the actual panorama
location returned by the Google Street View Metadata API.

The compiled dataset is later used to match Street View information with
cycling accident records.

Input
-----
Azure Blob Storage

Container:
    london

Folder:
    Accidents/Metadata/

Each metadata file is a JSON document containing:
    - Requested latitude and longitude
    - Retrieved panorama latitude and longitude
    - Street View imagery date (YYYY-MM)

Processing Steps
----------------
1. Connect to Azure Blob Storage.
2. Enumerate all metadata (.txt) files within the metadata folder.
3. Read each JSON file.
4. Extract the required metadata fields.
5. Store the extracted values in a list.
6. Convert the list into a pandas DataFrame.
7. Export the combined dataset as a CSV file.

Output
------
streetview_metadata_compiled.csv

Output Columns
--------------
requested_latitude
    Latitude originally supplied to the Street View Metadata API.

requested_longitude
    Longitude originally supplied to the Street View Metadata API.

retrieved_latitude
    Latitude of the panorama returned by Google Street View.

retrieved_longitude
    Longitude of the panorama returned by Google Street View.

time_utc
    Street View imagery date in YYYY-MM format.

Dependencies
------------
- pandas
- azure-storage-blob
- json
- tqdm
"""

In [ ]:
!pip install requests azure-storage-blob tqdm
import json
import pandas as pd
from azure.storage.blob import BlobServiceClient
from tqdm.auto import tqdm
# ==========================================================
# Azure Storage settings
# ==========================================================

CONNECTION_STRING = ""
CONTAINER_NAME = "london"

OUTPUT_CSV = "streetview_metadata_compiled.csv"

# ==========================================================
# Connect
# ==========================================================

blob_service = BlobServiceClient.from_connection_string(CONNECTION_STRING)
container = blob_service.get_container_client(CONTAINER_NAME)

rows = []

blob_count = 0
success = 0
failed = 0

# ==========================================================
# Read metadata files
# ==========================================================

blobs = list(container.list_blobs(name_starts_with="Accidents/Metadata/"))

for blob in tqdm(blobs, desc="Reading metadata"):
    if not blob.name.lower().endswith(".txt"):
        continue

    blob_count += 1

    try:
        blob_client = container.get_blob_client(blob)

        metadata = json.loads(
            blob_client.download_blob().readall().decode("utf-8")
        )

        response = metadata.get("metadata_response", {})
        location = response.get("location", {})

        rows.append({
            "requested_latitude": metadata.get("lat"),
            "requested_longitude": metadata.get("lng"),
            "retrieved_latitude": location.get("lat"),
            "retrieved_longitude": location.get("lng"),
            "time_utc": response.get("date")
        })

        success += 1

        if success % 1000 == 0:
            print(f"{success:,} metadata files processed")

    except Exception as e:
        failed += 1
        print(f"Failed: {blob.name}")
        print(e)

# ==========================================================
# Save CSV
# ==========================================================

df = pd.DataFrame(rows)

df.to_csv(OUTPUT_CSV, index=False)

print("\nDone.")
print(f"Metadata files found : {blob_count:,}")
print(f"Successfully read    : {success:,}")
print(f"Failed               : {failed:,}")
print(f"Rows written         : {len(df):,}")
print(f"Output               : {OUTPUT_CSV}")

In [ ]:
from google.colab import files
from azure.storage.blob import BlobServiceClient
import os

# ==========================================================
# Azure Storage settings
# ==========================================================

CONNECTION_STRING = ""
CONTAINER_NAME = "london"

# Folder in Azure Blob Storage
AZURE_FOLDER = "Accidents"

# ==========================================================
# Upload file from your computer
# ==========================================================


filename = "streetview_metadata_compiled.csv"

print(f"Uploading {filename}...")

# ==========================================================
# Connect to Azure
# ==========================================================

blob_service = BlobServiceClient.from_connection_string(CONNECTION_STRING)
container = blob_service.get_container_client(CONTAINER_NAME)

blob_path = f"{AZURE_FOLDER}/{os.path.basename(filename)}"

with open(filename, "rb") as data:
    container.upload_blob(
        name=blob_path,
        data=data,
        overwrite=True
    )

print("Upload complete!")
print(f"Blob path: {blob_path}")